In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

In [3]:
df = pd.read_csv("C:/Users/lerak/OneDrive/Desktop/bank_churn_analysis/data/raw/Bank_Customer_Churn_Prediction.csv")
df.head()

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Общая гипотеза: ожидаем, что наличие двух и более продуктов у клиента вызовет снижение оттока.

Нулевая гипотеза H0: количество продуктов не влияет на долю оттока. Доля оттока для групп A и B на самом деле не отличаются, и наблюдаемые различия случайны.

Альтернативная гипотеза H1: наличие двух и более продуктов снижает долю оттока. Доля оттока в группе B ниже, чем в группе A, и эти различия результат изменений.

Ожидаемый эффект: предполагаем, что доля оттока снизится с 25% до 12%, то есть более чем на 50%.

In [4]:
group_a = df[df['products_number'] == 1]['churn']
group_b = df[df['products_number'] >= 2]['churn']

print(f"Группа A : {len(group_a)} клиентов")
print(f"Группа B : {len(group_b)} клиентов")
print(f"\nДоля оттока в A: {group_a.mean():.4f}")
print(f"Доля оттока в B: {group_b.mean():.4f}")
print(f"Снижение оттока: {(group_a.mean() - group_b.mean()) / group_a.mean() * 100:.2f}%")

Группа A : 5084 клиентов
Группа B : 4916 клиентов

Доля оттока в A: 0.2771
Доля оттока в B: 0.1277
Снижение оттока: 53.91%


In [5]:
from statsmodels.stats.proportion import proportions_ztest

count = np.array([group_a.sum(), group_b.sum()])
nobs = np.array([len(group_a), len(group_b)])

z_stat, p_value = proportions_ztest(count, nobs)

print(f"Z-статистика: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nP-value < {alpha}. Отвергаем нулевую гипотезу.")
    print("Разница между группами статистически значима.")
else:
    print(f"\nP-value >= {alpha}. Не отвергаем нулевую гипотезу.")
    print("Разница между группами не значима.")

Z-статистика: 18.5447
P-value: 0.000000

P-value < 0.05. Отвергаем нулевую гипотезу.
Разница между группами статистически значима.


P-value меньше 0.05, разница между группами статистически значима. Клиенты с одним продуктом уходят значительно чаще, чем клиенты с двумя и более продуктами. Доля оттока в группе A составила 27.71%, в группе B 12.77%. Снижение оттока равно 53.91%, что близко к ожидаемому эффекту в 52%. Рекомендуется разработать программу cross-sell для клиентов с одним продуктом, чтобы снизить отток.